# SENet — Squeeze-and-Excitation Networks

**Paper:** Hu, J., Shen, L., & Sun, G. (2018).  
*Squeeze-and-Excitation Networks.*  
IEEE Conference on Computer Vision and Pattern Recognition (CVPR 2018).  
arXiv:1709.01507. [[PDF]](https://arxiv.org/abs/1709.01507)

**Award:** ILSVRC 2017 Classification Winner (2.25% top-5 error).

---

## Overview

SENet introduces **channel attention**: a lightweight mechanism that explicitly models
interdependencies between channels, allowing the network to recalibrate feature responses
by selectively emphasising informative channels and suppressing less useful ones.

The SE block has two operations:
1. **Squeeze**: Global average pooling collapses spatial dimensions into a channel descriptor.
2. **Excitation**: Two FC layers learn a soft channel weighting vector (in [0,1] via Sigmoid).

The channel weights then multiply the original feature map — **no spatial complexity added**.

## Architecture Diagrams (ASCII)

### SE Block
```
Feature map X [B, C, H, W]
        │
        ├─────────────────────────────────────┐  (identity)
        ▼                                     │
  GlobalAvgPool → [B, C, 1, 1]               │
        ▼                                     │
  Flatten → [B, C]                            │
        ▼                                     │
  FC(C → C//r) + ReLU  (r=16)                │
        ▼                                     │
  FC(C//r → C) + Sigmoid → s ∈ [0,1]^C       │
        ▼                                     │
  Reshape → [B, C, 1, 1]                      │
        └──────────────── × ──────────────────┘
                          ▼
               X̃ = X * s [B, C, H, W]
```

### SE Block inside ResNet Bottleneck
```
input ──────────────────────────────────── (+) ──→ ReLU
  │                                         ↑
  ▼                                         │
Conv 1×1 → BN → ReLU                       │
  ▼                                    (downsample if needed)
Conv 3×3 → BN → ReLU                       │
  ▼                                         │
Conv 1×1 → BN                              │
  ▼                                         │
SE Block (squeeze → excite → scale) ────────┘
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

BG   = '#0f1117'
FG   = '#e8eaf6'
SQZ  = '#42a5f5'  # squeeze
EXC  = '#ab47bc'  # excitation
SCL  = '#66bb6a'  # scale
BLK  = '#ef5350'  # block output

fig, axes = plt.subplots(1, 2, figsize=(14, 8), facecolor=BG)
fig.suptitle('SENet: Squeeze-and-Excitation Channel Attention', color=FG, fontsize=13, fontweight='bold')

# ── Left: SE Block detail ─────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(BG); ax.set_xlim(0, 6); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('SE Block Detail', color=FG, fontsize=11)

def box(ax, x, y, w, h, text, color, alpha=0.7):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1', fc=color, alpha=alpha, ec=FG, lw=1))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', color=FG, fontsize=8.5)

def arr(ax, x, y1, y2):
    ax.annotate('', xy=(x, y2), xytext=(x, y1), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

box(ax, 1.0, 8.8, 4, 0.7, 'Feature Map  X [B, C, H, W]', SQZ)
arr(ax, 3.0, 8.8, 8.1)
box(ax, 1.0, 7.3, 4, 0.7, 'GlobalAvgPool → [B, C, 1, 1]', SQZ, 0.5)
ax.text(0.5, 7.65, 'Squeeze', ha='center', color=SQZ, fontsize=8, fontstyle='italic', rotation=90)
arr(ax, 3.0, 7.3, 6.5)
box(ax, 1.0, 5.8, 4, 0.6, 'FC(C → C//r=16) + ReLU', EXC)
arr(ax, 3.0, 5.8, 5.0)
box(ax, 1.0, 4.3, 4, 0.6, 'FC(C//r → C) + Sigmoid', EXC, 0.5)
ax.text(0.5, 5.4, 'Excitation', ha='center', color=EXC, fontsize=8, fontstyle='italic', rotation=90)
arr(ax, 3.0, 4.3, 3.5)
box(ax, 1.0, 2.8, 4, 0.6, 'Scale vector s ∈ [0,1]^C', EXC, 0.3)

# Multiply arrow
ax.annotate('', xy=(4.5, 2.0), xytext=(3.0, 2.8), arrowprops=dict(arrowstyle='->', color=SCL, lw=1.5))
ax.annotate('', xy=(4.5, 2.0), xytext=(4.5, 8.8+0.35), arrowprops=dict(arrowstyle='->', color=FG, lw=1, linestyle='dashed'))
ax.annotate('', xy=(3.0, 1.5), xytext=(4.5, 2.0), arrowprops=dict(arrowstyle='->', color=SCL, lw=1.5))
ax.text(4.9, 5.0, '×\n(scale)', ha='center', color=SCL, fontsize=9)

box(ax, 1.0, 0.8, 4, 0.6, 'X̃ = X × s  [B, C, H, W]', SCL)
ax.text(0.5, 1.1, 'Scale', ha='center', color=SCL, fontsize=8, fontstyle='italic', rotation=90)

# ── Right: SE inside Bottleneck ───────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor(BG); ax2.set_xlim(0, 6); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('SE Block in ResNet Bottleneck', color=FG, fontsize=11)

RBLK = '#5c6bc0'
box(ax2, 1.0, 9.0, 4, 0.6, 'Input [B, C, H, W]', RBLK)
arr(ax2, 3.0, 9.0, 8.3)
box(ax2, 1.0, 7.6, 4, 0.6, 'Conv 1×1 + BN + ReLU', RBLK, 0.5)
arr(ax2, 3.0, 7.6, 6.9)
box(ax2, 1.0, 6.2, 4, 0.6, 'Conv 3×3 + BN + ReLU', RBLK, 0.5)
arr(ax2, 3.0, 6.2, 5.5)
box(ax2, 1.0, 4.8, 4, 0.6, 'Conv 1×1 + BN', RBLK, 0.5)
arr(ax2, 3.0, 4.8, 4.1)
box(ax2, 0.5, 3.4, 5, 0.6, 'SE Block (squeeze → excite → scale)', EXC)
ax2.text(3.0, 3.1, '↓  channel-wise recalibration', ha='center', color=EXC, fontsize=8)
arr(ax2, 3.0, 3.4, 2.7)

# Residual add
ax2.plot([1.0, 0.3, 0.3, 3.0], [9.3, 9.3, 2.3, 2.3], color=FG, lw=1, linestyle='dashed', alpha=0.6)
ax2.text(3.3, 2.35, '+ residual', color=FG, fontsize=8, alpha=0.7)
arr(ax2, 3.0, 2.7, 2.0)
box(ax2, 1.0, 1.3, 4, 0.6, 'ReLU → Output [B, 4C, H, W]', BLK)

plt.tight_layout()
plt.savefig('senet_diagram.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure saved.')

## Key Innovations & Mathematics

### Squeeze Operation

The squeeze aggregates spatial information:
$$z_c = F_{sq}(\mathbf{u}_c) = \frac{1}{H \times W} \sum_{i=1}^{H} \sum_{j=1}^{W} u_c(i, j)$$

where $\mathbf{u}_c \in \mathbb{R}^{H \times W}$ is the $c$-th channel feature map.

### Excitation Operation

The excitation learns non-linear channel dependencies:
$$\mathbf{s} = F_{ex}(\mathbf{z}, \mathbf{W}) = \sigma(\mathbf{W}_2 \delta(\mathbf{W}_1 \mathbf{z}))$$

where $\delta$ is ReLU, $\sigma$ is Sigmoid, $\mathbf{W}_1 \in \mathbb{R}^{C/r \times C}$, $\mathbf{W}_2 \in \mathbb{R}^{C \times C/r}$.

### Scale Operation

$$\tilde{\mathbf{x}}_c = F_{scale}(\mathbf{u}_c, s_c) = s_c \cdot \mathbf{u}_c$$

### Parameter Overhead

SE block parameters: $\frac{2C^2}{r}$ per block.  For $C=256, r=16$: 8,192 parameters.  
SE-ResNet-50 adds ~2.5M params (~10%) over baseline ResNet-50.

### Reduction Ratio r

| r | Top-1 (ImageNet) | SE Params |
|---|-----------------|----------|
| 2 | 77.4% | +50% |
| 4 | 77.6% | +25% |
| **16** | **77.6%** | **+10%** |
| 32 | 77.4% | +5% |

r=16 is the optimal trade-off chosen in the paper.

In [ ]:
# ── Code Walkthrough ──────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '../../src')
import torch
from hale_core.nn.vision.cnn.senet import SEBlock, SEBottleneck

# 1. SE Block behaviour
se = SEBlock(channels=256, reduction=16)
x  = torch.randn(2, 256, 14, 14)
y  = se(x)
print('SEBlock(256, r=16):')
print(f'  Input:  {tuple(x.shape)}')
print(f'  Output: {tuple(y.shape)}  (same shape — only channel magnitudes change)')

# Show scale values are in [0,1]
with torch.no_grad():
    scale = torch.sigmoid(se.fc2(torch.relu(se.fc1(se.pool(x)))))
print(f'  Scale range: [{scale.min():.4f}, {scale.max():.4f}] ← should be in [0,1]')

# 2. SEBottleneck
blk = SEBottleneck(in_planes=256, planes=64)  # expansion → 256
print(f'\nSEBottleneck(256, 64):  params = {sum(p.numel() for p in blk.parameters()):,}')

In [ ]:
# ── Usage Example ─────────────────────────────────────────────────────────────
from hale_core.nn.vision import build_vision_backbone

for name in ['se_resnet50', 'se_resnet101', 'seresnext50_32x4d']:
    b = build_vision_backbone(name)
    b.eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = b(x)
    print(f'{name}:')
    print(f'  pooled: {tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params: {b.num_parameters():,}')

## References

| # | Citation |
|---|----------|
| [1] | Hu, J., Shen, L., & Sun, G. (2018). **Squeeze-and-Excitation Networks.** CVPR. arXiv:1709.01507. |
| [2] | He, K., Zhang, X., Ren, S., & Sun, J. (2016). **Deep Residual Learning for Image Recognition.** CVPR. arXiv:1512.03385. |
| [3] | Woo, S., Park, J., Lee, J.-Y., & Kweon, I. S. (2018). **CBAM: Convolutional Block Attention Module.** ECCV. |
| [4] | Li, X., Wang, W., Hu, X., & Yang, J. (2019). **Selective Kernel Networks.** CVPR. |

---
*Notebook: SENet / SE-ResNet — HaleBlocks backbone family*  
*Architecture: `hale_core.nn.vision.cnn.senet`*